In [1]:
import pandas as pd

# Proyecto: 

## Justificación del problema

La empresa M&S cuenta con un aplicativo web para la gestión de productos químicos. En la base de datos, se cuenta con información relacionada a los productos directamente relacionada con la información de las Fichas de Datos de Seguridad (FDS), que hace la función de la hoja de vida del producto. Debido a la gran variabilidad en la forma en que estos pueden ser nombrados, se tiene un problema actual en donde se cuenta con aproximadamente el 7% de productos repetidos y tiempos altos en la consolidación de inventarios. 

Los clientes que usan el aplicativo, cuentan con información relacionada al nombre interno que la organización destina para el producto. De esta manera, un mismo producto puede aparecer registrado con múltiples variantes de nombre, diferencias en abreviaturas, idioma, formato o referencia comercial. Por ejemplo, un mismo producto puede registrarse como:

* NaOH 0.1N Cleaning Solution

* 0.1 N Sodium Hydroxide Solution

* NaOH 0.1N cuvette cleaning solution

Ante este contexto, es de alta importancia desarrollar mecanismos que permitan identificar automáticamente qué productos del catálogo existente son más similares a un nuevo registro, a partir de la información disponible como el nombre del producto y su fabricante.

Un sistema que sugiera coincidencias probables permitiría:

* reducir la creación de duplicados,

* mejorar la calidad y consistencia del catálogo de productos,

* facilitar la búsqueda y recuperación de información dentro del sistema,

* optimizar procesos de gestión de inventarios químicos.

Por esta razón, resulta relevante explorar el uso de técnicas de aprendizaje automático y representaciones semánticas del texto que permitan calcular similitudes entre productos y sugerir automáticamente las coincidencias más probables dentro del catálogo existente.

## Objetivo
Desarrollar un modelo de Deep Learning, el cual con el nombre y fabricante de un producto químico ingresado por el usuario, identifique los productos más similares en el catálogo existente, devolviendo un ranking de coincidencias probables.

## Metodología

1. Estructura inicial de los datos

El dataset de productos se divide en dos subconjuntos dependiendo de la disponibilidad de información adicional.

* df_con_sin_o_uso: Productos que tienen información en uso o sinónimo. Tamaño aproximado: (50621, 5)

* df_sin_sin_o_uso: Productos que NO tienen información ni en sinonimos ni en uso. Tamaño aproximado: (13874, 5)

La motivación de la separación nace porque los productos que tienen más información contextual permiten construir mejores representaciones semánticas para el modelo.

Sin embargo, ambos datasets siguen siendo parte del mismo catálogo y pueden utilizarse en el sistema final.

2. Limpieza y normalización de texto

Se aplican transformaciones para estandarizar el texto:

- convertir texto a minúsculas
- eliminar caracteres especiales innecesarios
- eliminar espacios duplicados
- reemplazar valores nulos por cadenas vacías


## Carga de datos

In [2]:
df = pd.read_excel("listado productos.xlsx").iloc[:, 1:]
df

,id,nombre_producto,sinonimos,uso,fabricante
0,30818,0 1 m dtt,superscript iv first strand synthesis system |...,para uso en investigacion unicamente,thermofisher scientific
1,56997,0 1n naoh compartimiento r1b,access hstni b52699,NaN,beckman coulter
2,10421,0 1n naoh cuvette cleaning solution b,7098 10445578,agentes de diagn oacute stico,siemens healthcare
3,42583,0 45 m 47mm white gridded s pak filter membran...,NaN,investigacion y analisis bioquimicos,merck
4,56156,0 5 medios de crecimiento b de postgate modifi...,NaN,NaN,grupo volante
...,...,...,...,...,...
64490,35967,zytiga,zytiga abiraterone acetate abiraterone acetate...,producto farmaceutico terminado grupo farmacot...,janssen
64491,1360,zytokil 10mg 5ml,doxorubicina,no registra,pisa farmaceutica
64492,26325,zz76 01009 citrus passion,NaN,NaN,ungerer & company
64493,62579,zzliteprop prime 108 alt code l426802 00,NaN,NaN,grupo transmerquim


In [3]:
# Separar el DataFrame en dos
# Uno con al menos un sinónimo o un uso
df_con_sin_o_uso = df[(df['sinonimos'].notna() & (df['sinonimos'] != '')) | (df['uso'].notna() & (df['uso'] != ''))]

# El otro con los demás
df_sin_sin_o_uso = df[~((df['sinonimos'].notna() & (df['sinonimos'] != '')) | (df['uso'].notna() & (df['uso'] != '')))]

print("DataFrame con sinónimo o uso:")
print(df_con_sin_o_uso.shape)
print("\nDataFrame sin sinónimo o uso:")
print(df_sin_sin_o_uso.shape)

DataFrame con sinónimo o uso:
(50621, 5)

DataFrame sin sinónimo o uso:
(13874, 5)


## Clustering